# Show SpecMine specs and features

Set one or more `REPO_IDS` or `SPEC_IDS` below (list or comma-separated string). A repository ID shows all distinct specs listed for that repository; a spec ID shows only that spec. Metadata and features are read from the local cache created by `initialize_data.ipynb`. The selected raw Markdown files are downloaded into `notebooks/data` (existing cached files are reused). Feature rows are joined by the exact `file_url_sha16` ID, and Markdown is escaped before display.

In [ ]:
import html
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src" / "specmine").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/specmine. Open this notebook from the project "
        "folder or one of its subfolders."
    )
sys.path.insert(0, str(project_root / "src"))

from specmine.data import download_spec_files, load_data, load_feature_data

DATA_DIR = project_root / "notebooks" / "data"
REPO_IDS = []  # Example: [123456789, 987654321] or "123456789, 987654321"
SPEC_IDS = []  # Example: ["0123456789abcdef", "fedcba9876543210"]


In [ ]:
def normalize_inputs(values):
    if values is None:
        return []
    if isinstance(values, str):
        values = values.split(",")
    elif not isinstance(values, (list, tuple, set)):
        values = [values]
    normalized = [str(value).strip() for value in values if str(value).strip()]
    return list(dict.fromkeys(normalized))


requested_repo_ids = normalize_inputs(REPO_IDS)
requested_spec_ids = normalize_inputs(SPEC_IDS)
if bool(requested_repo_ids) == bool(requested_spec_ids):
    raise ValueError("Set one or more REPO_IDS or SPEC_IDS, but not both.")

metadata = load_data(local_data_only=True, data_dir=DATA_DIR)
if metadata is None or metadata.empty:
    raise RuntimeError("Local SpecMine metadata is empty or unavailable.")

required_columns = {"repo_id", "repo_name", "file_url_sha16", "file_path", "file_url"}
missing_columns = required_columns - set(metadata.columns)
if missing_columns:
    raise KeyError(f"SpecMine metadata is missing columns: {sorted(missing_columns)}")

if requested_spec_ids:
    selected_specs = metadata.loc[
        metadata["file_url_sha16"].astype("string").isin(requested_spec_ids)
    ].copy()
    found_spec_ids = set(selected_specs["file_url_sha16"].astype("string"))
    missing_spec_ids = sorted(set(requested_spec_ids) - found_spec_ids)
    if missing_spec_ids:
        raise ValueError(f"No SpecMine specs found for spec_id(s): {missing_spec_ids}")
else:
    requested_repo_values = pd.to_numeric(
        pd.Series(requested_repo_ids), errors="coerce"
    )
    if (
        requested_repo_values.isna().any()
        or not requested_repo_values.mod(1).eq(0).all()
    ):
        raise ValueError("Every REPO_ID must be an integer repository ID.")
    normalized_repo_ids = {str(int(value)) for value in requested_repo_values}
    repository_ids = (
        pd.to_numeric(metadata["repo_id"], errors="coerce")
        .astype("Int64")
        .astype("string")
    )
    selected_specs = metadata.loc[repository_ids.isin(normalized_repo_ids)].copy()
    found_repo_ids = set(
        selected_specs["repo_id"].dropna().map(lambda value: str(int(value)))
    )
    missing_repo_ids = sorted(normalized_repo_ids - found_repo_ids)
    if missing_repo_ids:
        raise ValueError(f"No SpecMine specs found for repo_id(s): {missing_repo_ids}")

if selected_specs["file_url_sha16"].isna().any():
    raise ValueError("Selected metadata contains a spec with no spec_id.")
selected_specs = selected_specs.drop_duplicates(subset=["file_url_sha16"])
print(
    f"Selected {len(selected_specs)} distinct spec(s) from {selected_specs['repo_id'].nunique()} repository/repositories."
)
display(
    selected_specs[["repo_id", "repo_name", "file_url_sha16", "file_path"]]
    .sort_values(["repo_name", "file_path", "file_url_sha16"])
    .reset_index(drop=True)
)

In [ ]:
features = load_feature_data(
    selected_specs[["file_url_sha16"]],
    local_data_only=True,
    data_dir=DATA_DIR,
)
feature_ids = features["file_url_sha16"].astype("string")
selected_ids = selected_specs["file_url_sha16"].astype("string")
missing_feature_ids = sorted(set(selected_ids) - set(feature_ids))
if missing_feature_ids:
    raise ValueError(
        f"No precomputed feature row found for spec ID(s): {missing_feature_ids[:10]}"
    )
features = features.loc[feature_ids.isin(selected_ids)].copy()
if features["file_url_sha16"].astype("string").duplicated().any():
    raise ValueError("Feature table contains duplicate rows for a selected spec ID.")
if set(features["file_url_sha16"].astype("string")) != set(selected_ids):
    raise ValueError("Selected specs and feature rows do not match one-to-one.")

downloaded_specs = download_spec_files(selected_specs, data_dir=DATA_DIR)
downloaded_ids = set(downloaded_specs["file_url_sha16"].astype("string"))
if downloaded_ids != set(selected_ids):
    unavailable_ids = sorted(set(selected_ids) - downloaded_ids)
    raise RuntimeError(
        f"Could not download selected spec ID(s): {unavailable_ids[:10]}"
    )

spec_details = selected_specs.merge(
    features,
    on="file_url_sha16",
    how="inner",
    validate="one_to_one",
).merge(
    downloaded_specs[["file_url_sha16", "local_path"]],
    on="file_url_sha16",
    how="inner",
    validate="one_to_one",
)
if len(spec_details) != len(selected_specs):
    raise RuntimeError("Not every selected spec has exactly one feature row and file.")
print("Feature rows matched one-to-one by spec_id.")

In [ ]:
for row in spec_details.sort_values(
    ["repo_name", "file_path", "file_url_sha16"]
).itertuples(index=False):
    print(f"Repository: {row.repo_name} (repo_id={row.repo_id})")
    print(f"Spec file: {row.file_path}")
    print(f"spec_id: {row.file_url_sha16}")
    feature_values = features.loc[
        features["file_url_sha16"].astype("string").eq(str(row.file_url_sha16))
    ]
    display(HTML("<h4>Precomputed content features</h4>"))
    display(feature_values.reset_index(drop=True))
    spec_path = Path(row.local_path)
    if not spec_path.is_file():
        raise FileNotFoundError(f"Downloaded spec file is missing: {spec_path}")
    markdown = spec_path.read_text(encoding="utf-8")
    display(HTML("<h4>Spec Markdown</h4>"))
    display(HTML("<pre>" + html.escape(markdown) + "</pre>"))
    display(HTML("<hr>"))